### 1. Librerías y Carga de Datos
Cargamos las librerías necesarias para el procesamiento, modelado, visualización y la creación de la interfaz interactiva con Gradio.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Instalar Gradio de forma segura en una celda separada o aquí mismo
!pip install -q gradio
import gradio as gr

In [ ]:
# Cargar dataset
df_path = '/content/dataset_sistema_solar_sin_sol_2023_2026.csv'
df = pd.read_csv(df_path)

# Exploración básica
print("Dimensiones del dataset:", df.shape)
print("\nColumnas del dataset:")
print(df.info())
print("\nPrimeros registros:")
display(df.head())

### 2. Procesamiento de Datos y Entrenamiento de la Red Neuronal
Identificamos las variables predictoras, la variable objetivo (`especie` u homóloga en el dataset), realizamos la codificación, escalamiento, entrenamiento del modelo de clasificación y las predicciones correspondientes a fines de 2026.

In [ ]:
# Definir explícitamente la columna objetivo y temporal basadas en el dataset cargado
target_col = 'Grid_Status'
print(f"Columna objetivo identificada: {target_col}")

date_col = 'Timestamp'
df[date_col] = pd.to_datetime(df[date_col])

# Rellenar valores nulos si existen
df = df.ffill().bfill()

# Codificar variable objetivo
le = LabelEncoder()
y = le.fit_transform(df[target_col])
classes = le.classes_

# Seleccionar características numéricas para el entrenamiento (excluyendo la fecha y variables de salida no numéricas)
features = [col for col in df.columns if col not in [target_col, date_col] and df[col].dtype in [np.int64, np.float64]]
print(f"Características predictoras ({len(features)}): {features}")
X = df[features]

# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalamiento de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Construir Modelo de Red Neuronal
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(len(classes), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Entrenar
history = model.fit(
    X_train_scaled, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=32,
    verbose=1,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

### 3. Métricas de Clasificación e Histograma de Errores

In [ ]:
# Predicciones en Test
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred)

# Reporte de clasificación
clf_report = classification_report(y_test, y_pred, target_names=[str(c) for c in classes], output_dict=True)
clf_report_df = pd.DataFrame(clf_report).transpose()

# Histograma de Errores
errors = y_test - y_pred

# Generar y guardar las figuras para usarlas en Gradio
fig_metrics, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Matriz de Confusión')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

axes[1].hist(errors, bins=15, color='crimson', edgecolor='black', alpha=0.7)
axes[1].set_title('Histograma de Errores (Real - Predicho)')
axes[1].set_xlabel('Error')
axes[1].set_ylabel('Frecuencia')
plt.tight_layout()
fig_metrics.savefig('metricas_clasificacion.png')
plt.close()

### 4. Predicción de Noviembre y Diciembre 2026
Filtramos el dataset original para aislar las observaciones correspondientes a los últimos dos meses de 2026 y aplicamos la red neuronal entrenada para clasificar dichas muestras.

In [ ]:
# Filtrar datos de Noviembre y Diciembre de 2026
df_2026_nd = df[(df[date_col].dt.year == 2026) & (df[date_col].dt.month.isin([11, 12]))]

if not df_2026_nd.empty:
    X_nd = df_2026_nd[features]
    X_nd_scaled = scaler.transform(X_nd)
    preds_nd_probs = model.predict(X_nd_scaled)
    preds_nd = np.argmax(preds_nd_probs, axis=1)

    df_predictions = df_2026_nd.copy()
    df_predictions['Prediccion_Especie'] = le.inverse_transform(preds_nd)

    print("Ejemplos de predicciones para Noviembre/Diciembre 2026:")
    display(df_predictions[[date_col, 'Prediccion_Especie'] + features[:3]].head(10))
else:
    print("No se encontraron datos históricos correspondientes a Noviembre y Diciembre de 2026.")
    # Crear un escenario artificial/simulado en caso de no contar con las fechas
    df_predictions = pd.DataFrame()

### 5. Creación de Plots de EDA para la Interfaz de Gradio
Generamos las figuras univariadas, bivariadas, multivariadas y de correlación para integrarlas estáticamente o bajo demanda en las pestañas correspondientes de Gradio.

In [ ]:
# 1. Univariada
plt.figure(figsize=(8, 4))
sns.histplot(df[features[0]], kde=True, color='teal')
plt.title(f'Distribución de {features[0]} (Univariada)')
plt.tight_layout()
plt.savefig('eda_univariada.png')
plt.close()

# 2. Bivariada
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df.sample(min(2000, len(df))), x=features[0], y=features[1] if len(features) > 1 else features[0], hue=target_col, palette='viridis')
plt.title('Relación Bivariada')
plt.tight_layout()
plt.savefig('eda_bivariada.png')
plt.close()

# 3. Multivariada
plt.figure(figsize=(10, 5))
sns.pairplot(df.sample(min(500, len(df)))[features[:3] + [target_col]], hue=target_col, palette='coolwarm')
plt.tight_layout()
plt.savefig('eda_multivariada.png')
plt.close()

# 4. Matriz de Correlación
plt.figure(figsize=(10, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación de Variables')
plt.tight_layout()
plt.savefig('eda_correlacion.png')
plt.close()

### 6. Configuración y Lanzamiento del Dashboard de Gradio
Definimos la interfaz organizada por pestañas de acuerdo con la especificación completa del usuario.

In [ ]:
desc_columnas = """
### Descripción de Columnas y Metadatos
* **Timestamp (Fecha)**: Instante temporal de las lecturas energéticas.
* **P_DC_PV_W**: Potencia de Corriente Directa del generador fotovoltaico solar.
* **V_Batt_V e I_Batt_A**: Tensión y corriente de la batería del sistema.
* **Grid_Status (Objetivo/Especie)**: Estado operativo de la red eléctrica, utilizado como el target de clasificación del modelo.
"""

conclusiones_texto = """
### Conclusiones del Ejercicio
1. **Desempeño del Clasificador**: La red neuronal clasifica exitosamente el estado operacional del inversor solar (Grid_Status) usando parámetros puramente eléctricos.
2. **Perspectiva del EDA**: Las variables físicas como la potencia fotovoltaica y la temperatura del inversor se asocian de manera altamente correlacionada con los estados de falla y descarga de energía.
3. **Dashboard Unificado**: Gradio permite un acceso centralizado, veloz y sin interrupciones a todos los segmentos del pipeline de Ciencia de Datos.
"""

with gr.Blocks(title="Dashboard de Monitoreo - Sistema Solar") as demo:
    gr.Markdown("# 🌌 Dashboard Interactivo del Sistema Solar & Clasificador por Red Neuronal")

    with gr.Tab("Vista de Datos"):
        gr.Markdown("### Muestra del DataFrame Original")
        gr.DataFrame(df.head(20))

    with gr.Tab("Estadísticas Descriptivas"):
        gr.Markdown("### Resumen de Variables")
        gr.DataFrame(df.describe())
        gr.Markdown(desc_columnas)

    with gr.Tab("EDA Univariado"):
        gr.Image('eda_univariada.png', label="Distribución Univariada")

    with gr.Tab("EDA Bivariado"):
        gr.Image('eda_bivariada.png', label="Gráfico de Dispersión Bivariado")

    with gr.Tab("EDA Multivariado"):
        gr.Image('eda_multivariada.png', label="Pairplot Multivariado")

    with gr.Tab("Matriz de Correlación"):
        gr.Image('eda_correlacion.png', label="Coeficientes de Correlación")

    with gr.Tab("Modelo & Métricas"):
        gr.Markdown("### Métricas de Evaluación del Modelo")
        gr.DataFrame(clf_report_df)
        gr.Image('metricas_clasificacion.png', label="Matriz de Confusión e Histograma de Errores")

    with gr.Tab("Predicciones Nov/Dic 2026"):
        gr.Markdown("### Resultados de Predicción")
        if not df_2026_nd.empty:
            gr.DataFrame(df_predictions[[date_col, 'Prediccion_Especie'] + features[:3]].head(50))
        else:
            gr.Markdown("No hay registros temporales para Noviembre y Diciembre de 2026 en este dataset.")

    with gr.Tab("Explicación & Conclusiones"):
        gr.Markdown(conclusiones_texto)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://85086552a7187a964b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
